In [ ]:
!pip uninstall -y openai
!pip install -q -U openai
!pip install -q -U deepeval boto3 faiss-cpu sentence-transformers

import os
import re
import json
import random
import hashlib
import boto3
import faiss
import torch
import numpy as np
import pandas as pd

from functools import lru_cache
from google.colab import userdata
from botocore.exceptions import ClientError
from sentence_transformers import SentenceTransformer, CrossEncoder

In [ ]:
from google.colab import userdata
from deepeval.models import AzureOpenAIModel

endpoint = userdata.get("AZURE_OPENAI_ENDPOINT").strip().rstrip("/") + "/"
deployment = userdata.get("AZURE_OPENAI_DEPLOYMENT").strip()
model = userdata.get("AZURE_OPENAI_MODEL").strip()
api_key = userdata.get("AZURE_OPENAI_API_KEY").strip()

print("Endpoint:", endpoint)
print("Deployment:", deployment)
print("Model:", model)
print("API version: 2025-04-01-preview")

judge = AzureOpenAIModel(
    model=model,
    deployment_name=deployment,
    api_key=api_key,
    api_version="2025-04-01-preview",
    base_url=endpoint,
    temperature=0
)

print(judge.generate("Reply only with READY"))

Endpoint: https://legal-rag-practice.services.ai.azure.com/
Deployment: gpt-5.1
Model: gpt-5.1
API version: 2025-04-01-preview
('READY', 0.00011250000000000001)


In [ ]:
BUCKET = "legai-ai-embeddings"
FAISS_KEY = "faiss/faiss_rebuilt.index"
TRAIN_SAMPLE_KEY = "finetune/sampled_contexts_5000.jsonl"

LOCAL_FAISS = "/content/faiss_rebuilt.index"

EMBED_MODEL = "BAAI/bge-base-en-v1.5"
RERANK_MODEL = "BAAI/bge-reranker-base"

In [ ]:
aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID").strip()
aws_secret_access_key= ("AWS_SECRET_ACCESS_KEY").strip()

s3 = boto3.client("s3", aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID").strip(), aws_secret_access_key=userdata.get("AWS_SECRET_ACCESS_KEY").strip(), region_name="us-east-1")
print("S3 connected")

S3 connected


In [ ]:
s3.download_file(BUCKET, FAISS_KEY, LOCAL_FAISS)

index = faiss.read_index(LOCAL_FAISS)

print("FAISS vectors:", index.ntotal)
print("FAISS dimension:", index.d)

KeyboardInterrupt: 

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_model = SentenceTransformer(EMBED_MODEL, device=device)

reranker = CrossEncoder( RERANK_MODEL, max_length=512, device=device )

print("Device:", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Device: cuda


In [ ]:
print("Embedding model dimension:",
      embed_model.get_sentence_embedding_dimension())

print("FAISS index dimension:",
      index.d)

Embedding model dimension: 768
FAISS index dimension: 384


/tmp/ipykernel_1302/524986657.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_model.get_sentence_embedding_dimension())


In [ ]:
base_index = faiss.downcast_index(index.index)

print("Current efSearch:", base_index.hnsw.efSearch)

base_index.hnsw.efSearch = 256

print("New efSearch:", base_index.hnsw.efSearch)

Current efSearch: 64
New efSearch: 256


In [ ]:
def embed_query(question):

    embedding = embed_model.encode( [question], convert_to_numpy=True, normalize_embeddings=True ).astype("float32")

    embedding = embedding[:, :384]

    return np.ascontiguousarray(embedding)

In [ ]:
metadata_keys = []

for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix="metadata/"):

    metadata_keys.extend([obj["Key"] for obj in page.get("Contents", []) if obj["Key"].endswith(".jsonl")])

metadata_keys.sort(key=lambda x: int(x.split("metadata_")[-1].split(".jsonl")[0]))

print("Metadata files:", len(metadata_keys))

Metadata files: 7015


In [ ]:
@lru_cache(maxsize=128)
def load_metadata_file(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)

    return [json.loads(line) for line in obj["Body"].read().decode("utf-8").splitlines() if line.strip()]

In [ ]:
def get_metadata(vector_id):
    target = int(vector_id)
    left, right = 0, len(metadata_keys) - 1

    while left <= right:
        mid = (left + right) // 2
        rows = load_metadata_file(metadata_keys[mid])

        if not rows:
            return None

        first_id, last_id = int(rows[0]["vector_id"]), int(rows[-1]["vector_id"])

        if target < first_id:
            right = mid - 1
        elif target > last_id:
            left = mid + 1
        else:
            for row in rows:
                if int(row["vector_id"]) == target:
                    return row
            return None

    return None

In [ ]:
def retrieve_candidates(question, k=20):
    query_vector = embed_query(question)
    distances, ids = index.search(query_vector, k)
    results = []

    for distance, vector_id in zip(distances[0], ids[0]):

        if vector_id == -1:
            continue

        metadata = get_metadata(vector_id)

        if metadata:
            results.append({"vector_id": int(vector_id), "distance": float(distance), "source": metadata.get("source"), "source_file": metadata.get("source_file"), "text": metadata["text"]})

    return results

In [ ]:
def rerank(question, results):
    if not results:
        return []

    pairs = [[question, r["text"]] for r in results]
    scores = reranker.predict(pairs)

    for r, score in zip(results, scores):
        r["rerank_score"] = float(score)

    return sorted(results, key=lambda x: x["rerank_score"], reverse=True)

In [ ]:
def filter_evidence(results, min_score=0.90, max_docs=5):
    strong = [r for r in results if r["rerank_score"] >= min_score]
    return strong[:max_docs]

In [ ]:
def split_sentences(text): return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

In [ ]:
def get_best_evidence(question, results, top_n=3, min_score=0.90):

    sentences = []

    for i, r in enumerate(results, 1):
        for sentence in split_sentences(r["text"]):

            if len(sentence) > 30:
                sentences.append({"document": i, "text": sentence})

    if not sentences:
        return []

    scores = reranker.predict([[question, s["text"]] for s in sentences])

    for s, score in zip(sentences, scores):
        s["score"] = float(score)

    sentences = sorted(sentences, key=lambda x: x["score"], reverse=True)
    return [s for s in sentences if s["score"] >= min_score][:top_n]

In [ ]:
def retrieve(question, candidate_k=20, min_score=0.90, max_docs=5):
    candidates = retrieve_candidates(question, candidate_k)
    ranked = rerank(question, candidates)
    return filter_evidence(ranked, min_score, max_docs)

In [ ]:
def deduplicate_evidence(evidence, similarity_threshold=0.95):
    unique = []
    unique_embeddings = []

    for e in evidence:

        text = clean_evidence_text(e["text"])

        emb = embed_model.encode(
            text,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        duplicate = False

        for previous_embedding in unique_embeddings:

            similarity = float(
                np.dot(emb, previous_embedding)
            )

            if similarity >= similarity_threshold:
                duplicate = True
                break

        if not duplicate:
            item = e.copy()
            item["text"] = text

            unique.append(item)
            unique_embeddings.append(emb)

    return unique

In [ ]:
def is_multi_rule_question(question):
    q = question.lower()
    keywords = ["compare", "difference", "different", "both", " and ", "periods", "rules", "requirements"]
    return any(k in q for k in keywords)

def clean_evidence_text(text):
    text = text.replace("Source:", "").strip()
    text = text.replace("Effective Date Note:", "").strip()
    text = text.replace("Civilian", "").strip()
    return text

In [ ]:
def rag(question, candidate_k=20, min_score=0.90, max_docs=5):

    results = retrieve(
        question,
        candidate_k=candidate_k,
        min_score=min_score,
        max_docs=max_docs
    )

    if not results:
        return "The retrieved legal context does not provide enough information to answer this question.", []

    context = build_context(results)

    answer = ask_legai(
        context=context,
        question=question
    )

    return answer, results

In [ ]:
def text_hash(text):
  return hashlib.sha256( text.strip().encode("utf-8") ).hexdigest()

In [ ]:
def load_training_hashes():
    hashes = set()

    try:
        obj = s3.get_object(
            Bucket=BUCKET,
            Key=TRAIN_SAMPLE_KEY
        )

        lines = obj["Body"].read().decode("utf-8").splitlines()

        for line in lines:

            if not line.strip():
                continue

            row = json.loads(line)

            text = row.get("text") or row.get("context")

            if text:
                hashes.add(text_hash(text))

    except ClientError as e:

        code = e.response["Error"]["Code"]

        if code in ["NoSuchKey", "404"]:
            print("Training sample file was not found. No exclusion applied.")

        else:
            raise

    return hashes

In [ ]:
training_hashes = load_training_hashes()

print("Fine-tuning contexts excluded:", len(training_hashes))

Fine-tuning contexts excluded: 5000


In [ ]:
def sample_heldout_contexts(n=50, min_chars=200, max_chars=3500, seed=42):
    rng = random.Random(seed)

    keys = metadata_keys.copy()
    rng.shuffle(keys)

    contexts = []
    selected_info = []
    seen = set()

    for key in keys:

        rows = list(load_metadata_file(key))
        rng.shuffle(rows)

        for row in rows:

            text = row.get("text", "").strip()

            if len(text) < min_chars or len(text) > max_chars:
                continue

            h = text_hash(text)

            if h in training_hashes:
                continue

            if h in seen:
                continue

            seen.add(h)

            contexts.append([text])

            selected_info.append({
                "vector_id": row.get("vector_id"),
                "source": row.get("source"),
                "source_file": row.get("source_file"),
                "metadata_key": key
            })

            if len(contexts) >= n:
                return contexts, selected_info

    return contexts, selected_info

In [ ]:
N_CONTEXTS = 50

eval_contexts, eval_context_info = sample_heldout_contexts(
    n=N_CONTEXTS
)

print("Held-out evaluation contexts:", len(eval_contexts))

Held-out evaluation contexts: 50


In [ ]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import StylingConfig

styling = StylingConfig(
    task="Evaluate a legal retrieval-augmented question answering system.",
    scenario="A user asks factual questions that should be answered only from supplied legal material.",
    input_format="A clear legal question whose answer is explicitly supported by the supplied context.",
    expected_output_format="A concise factual answer grounded only in the supplied context. Preserve exact numbers, dates, time periods, section numbers, parties, actions, and legal conditions."
)

In [ ]:
synthesizer = Synthesizer( model=judge, styling_config=styling, async_mode=False )

goldens = synthesizer.generate_goldens_from_contexts(
    contexts=eval_contexts,
    include_expected_output=True,
    max_goldens_per_context=1
)

print("Generated goldens:", len(goldens))

Output()

Generated goldens: 50


In [ ]:
# import json
# from types import SimpleNamespace

# GOLDENS_FILE = "/content/legai_automated_goldens.jsonl"

# goldens = []

# with open(GOLDENS_FILE, "r", encoding="utf-8") as f:
#     for line in f:
#         row = json.loads(line)

#         goldens.append(
#             SimpleNamespace(
#                 input=row["input"],
#                 expected_output=row["expected_output"],
#                 context=row["context"]
#             )
#         )

# print("Loaded fixed goldens:", len(goldens))

Loaded fixed goldens: 50


In [ ]:
for i, golden in enumerate(goldens[:5], 1):
    print(f"\n========== GOLDEN {i} ==========")

    print("\nQUESTION:")
    print(golden.input)

    print("\nEXPECTED ANSWER:")
    print(golden.expected_output)

    print("\nSOURCE CONTEXT:")
    print(golden.context[0][:1200] if golden.context else "No context")

In [ ]:
import json

FIXED_EVAL_FILE = "/content/legai_fixed_eval_50.jsonl"

with open(FIXED_EVAL_FILE, "w", encoding="utf-8") as f:

    for i, golden in enumerate(goldens):

        info = eval_context_info[i]

        record = {
            "test_case": i,
            "input": golden.input,
            "expected_output": golden.expected_output,
            "context": golden.context,
            "vector_id": int(info["vector_id"]),
            "source": info.get("source"),
            "source_file": info.get("source_file"),
            "metadata_key": info.get("metadata_key")
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", FIXED_EVAL_FILE)

Saved: /content/legai_fixed_eval_50.jsonl


In [ ]:
from deepeval.test_case import LLMTestCase

test_cases = []
rag_run_records = []

for i, golden in enumerate(goldens, 1):

    question = golden.input

    print(f"Running {i}/{len(goldens)}: {question[:100]}")

    try:

        answer, evidence = rag(question)

        retrieval_context = [ clean_evidence_text(e["text"]) for e in evidence ]

        if not retrieval_context: retrieval_context = [ "NO_RELEVANT_CONTEXT_RETRIEVED" ]

        test_case = LLMTestCase( input=question, actual_output=answer, expected_output=golden.expected_output, retrieval_context=retrieval_context, context=golden.context )

        test_cases.append(test_case)

        rag_run_records.append({
            "question": question,
            "expected_answer": golden.expected_output,
            "actual_answer": answer,
            "retrieval_context": retrieval_context,
            "source_context": golden.context
        })

    except Exception as e:

        print("ERROR:", e)

Running 1/50: Under the applicable legal materials, at what time is posting ordinarily effective for a non‑investm
Running 2/50: Under the applicable legal rules governing Thrift Savings Plan (TSP) contributions and corrective ac
Running 3/50: How does a law that grants a single agency sole authority over confidential statistical data limit o
Running 4/50: How does federal law formally designate and classify a government position that controls more than $
Running 5/50: Under §1631.8, if an agency’s search for records exceeds the prescribed deadlines, what specific man
Running 6/50: Under the applicable legal framework, how do pay caps for senior executives differ when an employer 
Running 7/50: What remedial standards apply when a Recognized Statistical Unit repeatedly violates 44 U.S.C. § 356
Running 8/50: How are non-rollover distributions from eligible employer plans taxed compared to Roth funds, under 
Running 9/50: What are the legal and financial consequences under federal retire

In [ ]:
print("Test cases ready:", len(test_cases))

Test cases ready: 50


In [ ]:
from deepeval.test_case import LLMTestCase

retrieval_test_cases = []

for i, golden in enumerate(goldens, 1):

    question = golden.input

    print(f"Running retrieval {i}/{len(goldens)}: {question[:100]}")

    try:
        candidates = retrieve_candidates(question, k=20)

        ranked = rerank(question, candidates)

        retrieval_context = [
            clean_evidence_text(r["text"])
            for r in ranked[:5]
        ]

        if not retrieval_context:
            retrieval_context = ["NO_RELEVANT_CONTEXT_RETRIEVED"]

        test_case = LLMTestCase(
            input=question,
            actual_output="Retrieval evaluation only",
            expected_output=golden.expected_output,
            retrieval_context=retrieval_context,
            context=golden.context
        )

        retrieval_test_cases.append(test_case)

    except Exception as e:
        print("ERROR:", e)

print("Retrieval test cases:", len(retrieval_test_cases))

Running retrieval 1/50: Under the applicable legal materials, at what time is posting ordinarily effective for a non‑investm
Running retrieval 2/50: Under the applicable legal rules governing Thrift Savings Plan (TSP) contributions and corrective ac
Running retrieval 3/50: How does a law that grants a single agency sole authority over confidential statistical data limit o
Running retrieval 4/50: How does federal law formally designate and classify a government position that controls more than $
Running retrieval 5/50: Under §1631.8, if an agency’s search for records exceeds the prescribed deadlines, what specific man
Running retrieval 6/50: Under the applicable legal framework, how do pay caps for senior executives differ when an employer 
Running retrieval 7/50: What remedial standards apply when a Recognized Statistical Unit repeatedly violates 44 U.S.C. § 356
Running retrieval 8/50: How are non-rollover distributions from eligible employer plans taxed compared to Roth funds, under 


In [ ]:
RAG_OUTPUT_FILE = "/content/legai_rag_outputs.jsonl"

with open(RAG_OUTPUT_FILE, "w", encoding="utf-8") as f:

    for row in rag_run_records:
        f.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )

print("Saved:", RAG_OUTPUT_FILE)

Saved: /content/legai_rag_outputs.jsonl


In [ ]:
import os

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "180"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "2"

In [ ]:
from deepeval.metrics import ( ContextualPrecisionMetric, ContextualRecallMetric, ContextualRelevancyMetric )

retrieval_metrics = [
    ContextualPrecisionMetric( model=judge, threshold=0.7, include_reason=True, async_mode=False ),

    ContextualRecallMetric(model=judge,threshold=0.7,include_reason=True,async_mode=False),

    ContextualRelevancyMetric(model=judge,threshold=0.7,include_reason=True,async_mode=False)]

In [ ]:
# from deepeval import evaluate
# from deepeval.evaluate import AsyncConfig, DisplayConfig, ErrorConfig

# retrieval_results = evaluate(
#     test_cases=test_cases,
#     metrics=retrieval_metrics,
#     identifier="LegaiAI-Retrieval-Evaluation",
#     hyperparameters={
#         "embedding_model": EMBED_MODEL,
#         "reranker": RERANK_MODEL,
#         "candidate_k": 20,
#         "document_threshold": 0.90,
#         "sentence_threshold": 0.90
#     },
#     async_config=AsyncConfig(
#         run_async=False
#     ),
#     error_config=ErrorConfig(
#         ignore_errors=True
#     ),
#     display_config=DisplayConfig(
#         results_folder="/content/legai_eval_retrieval"
#     )
# )

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig, ErrorConfig

retrieval_results_v2 = evaluate(
    test_cases=retrieval_test_cases,
    metrics=retrieval_metrics,
    identifier="LegaiAI-Retrieval-efSearch256-k50",
    hyperparameters={
        "embedding_model": EMBED_MODEL,
        "reranker": RERANK_MODEL,
        "candidate_k": 20,
        "reranked_top_k": 5,
        "efSearch": 256,
        "rerank_threshold": "none"
    },
    async_config=AsyncConfig(
        run_async=False
    ),
    error_config=ErrorConfig(
        ignore_errors=True
    ),
    display_config=DisplayConfig(
        results_folder="/content/legai_eval_retrieval_ef256"
    )
)

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-5.1 (Azure), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-5.1 (Azure), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-5.1 (Azure), strict=False, 
async_mode=False)...

Output()

ERROR:deepeval.retry.azure:call timed out after 180s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt. Retrying: 1 time(s)...
INFO:deepeval.retry.azure:Retrying in 1.8160335369605953 s (attempt 1) after TimeoutError('call timed out after 180s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt.')
ERROR:deepeval.retry.azure:call timed out after 180s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt. Retrying: 1 time(s)...
INFO:deepeval.retry.azure:Retrying in 2.4483116826256888 s (attempt 1) after TimeoutError('call timed out after 180s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt.')
ERROR:deepeval.retry.azure:call timed out after 180s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              Under the applicable legal materials, at what time is posting ordinarily             │
│  │                         effective for a non‑investment request submitted via the TSP website at 11:45        │
│  │                         a.m. ET?                                                                             │
│  │     Actual Output:      Retrieval evaluation only                                                            │
│  │     Expected Output:    For a non‑investment transaction request submitted via the TSP website at 11:45      │
│  │                         a.m. Eastern Time on a business day, posting is ordinarily effective **that same     │
│  │                         business day**, because it was entered before 12 noon ET.                            │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Contextual Precision │ 1.00  │ 0.70      │ The score is 1.00 because the first two nodes i...    │
│        PASS  │ Contextual Recall    │ 1.00  │ 0.70      │ The score is 1.00 because the single sentence i...    │
│        FAIL  │ Contextual Relevancy │ 0.50  │ 0.70      │ The score is 0.50 because, although most of the       │
│              │                      │       │           │ retrieval context discusses unrelated topics like     │
│              │                      │       │           │ "a registered clearing agency" posting "manuals,      │
│              │                      │       │           │ notices, circulars" and disclosure of "the            │
│              │                      │       │           │ company's complete portfolio holdings" under          │
│              │                      │       │           │ Regulation S-X, there are a few directly relevant     │
│              │                      │       │           │ statements that clearly answer the timing             │
│              │                      │       │           │ question—specifically, "A transaction request other   │
│              │                      │       │           │ than an investment election request entered into      │
│              │                      │       │           │ the TSP record keeping system... before 12 noon       │
│              │                      │       │           │ eastern time... will ordinarily be posted that        │
│              │                      │       │           │ business day" and "A transaction request other than   │
│              │                      │       │           │ an investment election request entered into the       │
│              │                      │       │           │ system at or after 12 noon eastern time... will       │
│              │                      │       │           │ ordinarily be posted on the next business day."       │
│              │                      │       │           │ These relevant sentences are mixed in with largely    │
│              │                      │       │           

⚠ WARNING: No prompts logged.
» ]8;id=692962;https://deepeval.com/docs/evaluation-prompts\Log prompts]8;;\ to evaluate and optimize your prompt templates and models.

================================================================================

Test run saved at /content/legai_eval_retrieval_ef256/test_run_20260925_204007.json

✓ Evaluation completed 🎉! (time taken: 1814.1s | token cost: 1.8055299999999996 USD)
» Test Results (50 total tests):
   » Pass Rate: 54.0% | Passed: 27 | Failed: 23

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    GEval
)

from deepeval.test_case import SingleTurnParams

legal_correctness = GEval(
    name="Legal Correctness",
    evaluation_steps=[
        "Compare the actual output with the expected output.",
        "Check whether all material legal facts are correct.",
        "Treat any changed number, date, duration, notice period, section number, party, action, trigger, or legal condition as a material error.",
        "Penalize an answer that changes who performs an action or when the action becomes effective.",
        "Penalize unsupported additions or invented legal requirements.",
        "Penalize omission of information necessary to correctly answer the user's question."
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT
    ],
    model=judge,
    threshold=0.8,
    async_mode=False
)

In [ ]:
answer_metrics = [
    AnswerRelevancyMetric(
        model=judge,
        threshold=0.7,
        include_reason=True,
        async_mode=False
    ),

    FaithfulnessMetric(
        model=judge,
        threshold=0.7,
        include_reason=True,
        async_mode=False
    ),

    legal_correctness
]

In [ ]:
answer_results = evaluate(
    test_cases=test_cases,
    metrics=answer_metrics,
    identifier="LegaiAI-Answer-Evaluation",
    hyperparameters={
        "system": "LegaiAI",
        "answer_mode": "retrieval + reranking + fine-tuned LLM generation",
        "judge_model": model
    },
    async_config=AsyncConfig(
        run_async=False
    ),
    error_config=ErrorConfig(
        ignore_errors=True
    ),
    display_config=DisplayConfig(
        results_folder="/content/legai_eval_answers"
    )
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.1 (Azure), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-5.1 (Azure), strict=False, async_mode=False)...

✨ You're running DeepEval's latest Legal Correctness [GEval] Metric! (using gpt-5.1 (Azure), strict=False, 
async_mode=False)...

Output()

ERROR:deepeval.retry.azure:call timed out after 180s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt. Retrying: 1 time(s)...
INFO:deepeval.retry.azure:Retrying in 1.6455545928185746 s (attempt 1) after TimeoutError('call timed out after 180s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt.')


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              Under the applicable legal materials, at what time is posting ordinarily             │
│  │                         effective for a non‑investment request submitted via the TSP website at 11:45        │
│  │                         a.m. ET?                                                                             │
│  │     Actual Output:      The retrieved legal context does not provide enough information to answer this       │
│  │                         question.                                                                            │
│  │     Expected Output:    For a non‑investment transaction request submitted via the TSP website at 11:45      │
│  │                         a.m. Eastern Time on a business day, posting is ordinarily effective **that same     │
│  │                         business day**, because it was entered before 12 noon ET.                            │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                    ┃ Score ┃ Threshold ┃ Reason                                           │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy          │ 1.00  │ 0.70      │ The score is 1.00 because the answer is fully    │
│              │                           │       │           │ f...                                             │
│        PASS  │ Faithfulness              │ 1.00  │ 0.70      │ The score is 1.0 because there are no listed     │
│              │                           │       │           │ co...                                            │
│        FAIL  │ Legal Correctness [GEval] │ 0.10  │ 0.80      │ The actual output fails to address the core      │
│              │                           │       │           │ legal question and does not mention the          │
│              │                           │       │           │ effective posting time or the same‑day           │
│              │                           │       │           │ business posting rule before 12 noon ET. It      │
│              │                           │       │           │ omits the key material facts about when          │
│              │                           │       │           │ posting is ordinarily effective and instead      │
│              │                           │       │           │ claims insufficient information, contradicting   │
│              │                           │       │           │ the expected clear answer based on the legal     │
│              │                           │       │           │ materials. There are no invented legal           │
│              │                           │       │           │ requirements, but the necessary information to   │
│              │                           │       │           │ correctly answer the user's question is          │
│              │                           │       │           │ entirely missing, so it does not align with      │
│              │                           │       │           │ the evaluation steps.                            │
│                                                         

⚠ WARNING: No prompts logged.
» ]8;id=221983;https://deepeval.com/docs/evaluation-prompts\Log prompts]8;;\ to evaluate and optimize your prompt templates and models.

================================================================================

Test run saved at /content/legai_eval_answers/test_run_20260925_205401.json

✓ Evaluation completed 🎉! (time taken: 833.71s | token cost: 0.55069875 USD)
» Test Results (50 total tests):
   » Pass Rate: 22.0% | Passed: 11 | Failed: 39

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
!find /content/legai_eval_retrieval -type f -maxdepth 1 -print
!find /content/legai_eval_answers -type f -maxdepth 1 -print

find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
find: ‘/content/legai_eval_retrieval’: No such file or directory
find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/content/legai_eval_answers/test_run_20260925_205401.json
/content/legai_eval_answers/.test_run.lock


In [ ]:
# diagnostics = []

# for i, golden in enumerate(goldens):

#     question = golden.input
#     expected_vector_id = int(eval_context_info[i]["vector_id"])

#     # FAISS top 50
#     candidates = retrieve_candidates(question, k=20)

#     faiss_ids = [r["vector_id"] for r in candidates]

#     if expected_vector_id in faiss_ids:
#         faiss_rank = faiss_ids.index(expected_vector_id) + 1
#     else:
#         faiss_rank = None

#     # CrossEncoder reranking
#     ranked = rerank(question, candidates)

#     reranked_ids = [r["vector_id"] for r in ranked]

#     if expected_vector_id in reranked_ids:
#         rerank_rank = reranked_ids.index(expected_vector_id) + 1
#     else:
#         rerank_rank = None

#     diagnostics.append({
#         "test_case": i,
#         "expected_vector_id": expected_vector_id,
#         "faiss_rank": faiss_rank,
#         "rerank_rank": rerank_rank,
#         "in_faiss_top50": faiss_rank is not None,
#         "in_reranked_top5": (
#             rerank_rank is not None and rerank_rank <= 5
#         )
#     })

# df_diag = pd.DataFrame(diagnostics)

# display(df_diag)

# print("\nFAISS source hit @50:")
# print(df_diag["in_faiss_top50"].mean())

# print("\nCorrect source in reranked top 5:")
# print(df_diag["in_reranked_top5"].mean())

# print("\nCorrect source missing from FAISS top 50:")
# display(
#     df_diag[df_diag["in_faiss_top50"] == False]
# )

# print("\nFAISS found source, but reranker pushed it below top 5:")
# display(
#     df_diag[
#         (df_diag["in_faiss_top50"] == True) &
#         (df_diag["in_reranked_top5"] == False)
#     ]
# )

In [ ]:
print("Correct source in reranked top 3:")

print(
    (
        df_diag["rerank_rank"].notna() &
        (df_diag["rerank_rank"] <= 3)
    ).mean()
)

Correct source in reranked top 3:
0.6
